In [ ]:
!pip install -q ucimlrepo

In [ ]:
#import libraries section
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset
from ucimlrepo import fetch_ucirepo

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Saving artifacts
import joblib

# Warnings
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
#	Load dataset
# Fetch UCI dataset
dataset = fetch_ucirepo(id=697)

# Features and target
X = dataset.data.features.copy()
y = dataset.data.targets.copy()

# Clean column names
X.columns = X.columns.str.strip()
y.columns = y.columns.str.strip()

# Clean target values
y["Target"] = y["Target"].str.strip()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

Features shape: (4424, 36)
Target shape: (4424, 1)


,Marital Status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0,0.000000,0,10.8,1.4,1.74
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,0,6,0,0,0.000000,0,10.8,1.4,1.74
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79


,Target
0,Dropout
1,Graduate
2,Dropout
3,Graduate
4,Graduate


In [ ]:
#for checking loaded datasets
print("Feature columns:")
for column in X.columns:
    print("-", column)

print("\nTarget classes:")
print(y["Target"].value_counts())

Feature columns:
- Marital Status
- Application mode
- Application order
- Course
- Daytime/evening attendance
- Previous qualification
- Previous qualification (grade)
- Nacionality
- Mother's qualification
- Father's qualification
- Mother's occupation
- Father's occupation
- Admission grade
- Displaced
- Educational special needs
- Debtor
- Tuition fees up to date
- Gender
- Scholarship holder
- Age at enrollment
- International
- Curricular units 1st sem (credited)
- Curricular units 1st sem (enrolled)
- Curricular units 1st sem (evaluations)
- Curricular units 1st sem (approved)
- Curricular units 1st sem (grade)
- Curricular units 1st sem (without evaluations)
- Curricular units 2nd sem (credited)
- Curricular units 2nd sem (enrolled)
- Curricular units 2nd sem (evaluations)
- Curricular units 2nd sem (approved)
- Curricular units 2nd sem (grade)
- Curricular units 2nd sem (without evaluations)
- Unemployment rate
- Inflation rate
- GDP

Target classes:
Target
Graduate    2209
Dr

In [ ]:
# Define prediction scenario
PREDICTION_SCENARIO = "after_first_semester"

# Remove second semester information
second_semester_columns = [
    col for col in X.columns
    if "2nd sem" in col.lower()
]

X = X.drop(columns=second_semester_columns)

print("Removed columns:", second_semester_columns)
print("Remaining features:", X.shape[1])

Removed columns: ['Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)']
Remaining features: 30


In [ ]:
# Handle missing values
# Check missing values
missing_values = X.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage": (missing_values / len(X)) * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)
display(missing_summary[missing_summary["Missing Values"] > 0])

,Missing Values,Percentage


In [ ]:
#Identify feature types
# Categorical columns
categorical_features = [
    "Marital Status",
    "Application mode",
    "Course",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation"
]

# Binary columns
binary_features = [
    "Daytime/evening attendance",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

# Validate column names
for col in categorical_features + binary_features:
    if col not in X.columns:
        raise ValueError(f"Column not found: {col}")

# Remaining columns are numerical
numerical_features = [
    col for col in X.columns
    if col not in categorical_features + binary_features
]

print("Categorical:", len(categorical_features))
print("Binary:", len(binary_features))
print("Numerical:", len(numerical_features))

print("\nNumerical features:")
print(numerical_features)

Categorical: 9
Binary: 8
Numerical: 13

Numerical features:
['Application order', 'Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP']


In [ ]:
#Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y["Target"],
    test_size=0.20,
    random_state=42,
    stratify=y["Target"] #we use this because protect class distribution
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (3539, 30)
Testing set: (885, 30)


In [12]:
#test -> comparing class distriution
distribution = pd.DataFrame({
    "Original": y["Target"].value_counts(normalize=True),
    "Training": y_train.value_counts(normalize=True),
    "Testing": y_test.value_counts(normalize=True)
})

display(distribution.round(3))

,Original,Training,Testing
Target,,,
Graduate,0.499,0.499,0.499
Dropout,0.321,0.321,0.321
Enrolled,0.179,0.179,0.180


In [13]:
# preprocessing pipeline
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])
binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
        ("bin", binary_pipeline, binary_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created!")

Preprocessing pipeline created!


In [14]:
#running pipeline
# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using training parameters
X_test_processed = preprocessor.transform(X_test)

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Convert to DataFrames
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Training data after preprocessing:")
print(X_train_processed.shape)

print("\nTesting data after preprocessing:")
print(X_test_processed.shape)

display(X_train_processed.head())

Training data after preprocessing:
(3539, 230)

Testing data after preprocessing:
(885, 230)


,num__Application order,num__Previous qualification (grade),num__Admission grade,num__Age at enrollment,num__Curricular units 1st sem (credited),num__Curricular units 1st sem (enrolled),num__Curricular units 1st sem (evaluations),num__Curricular units 1st sem (approved),num__Curricular units 1st sem (grade),num__Curricular units 1st sem (without evaluations),...,cat__Father's occupation_194,cat__Father's occupation_195,bin__Daytime/evening attendance,bin__Displaced,bin__Educational special needs,bin__Debtor,bin__Tuition fees up to date,bin__Gender,bin__Scholarship holder,bin__International
2283,-0.556229,-0.960935,-0.569684,-0.295828,3.481269,1.898375,1.363422,1.711901,0.364851,-0.210957,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3874,-0.556229,0.031460,-0.473323,2.334335,-0.301418,-0.507336,-0.786079,-1.519082,-2.181071,1.297276,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
2281,-0.556229,0.554172,0.903267,-0.295828,-0.301418,-0.106384,1.124589,-0.226689,0.231395,-0.210957,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
817,-0.556229,-0.582158,-0.418259,-0.690353,-0.301418,-0.106384,-0.547245,0.419507,0.796015,-0.210957,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
404,-0.556229,0.705683,0.304450,-0.558845,-0.301418,0.695519,-0.069578,0.742606,0.843531,-0.210957,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [15]:
# missing values check
print("Training missing values:")
print(X_train_processed.isnull().sum().sum())

print("\nTesting missing values:")
print(X_test_processed.isnull().sum().sum())
# we expecting -> both 0

Training missing values:
0

Testing missing values:
0


In [16]:
#scaling check#
# Check numerical feature means
numerical_means = X_train_processed[
    [f"num__{col}" for col in numerical_features]
].mean()

print(numerical_means.round(3))

num__Application order                                 0.0
num__Previous qualification (grade)                    0.0
num__Admission grade                                  -0.0
num__Age at enrollment                                 0.0
num__Curricular units 1st sem (credited)              -0.0
num__Curricular units 1st sem (enrolled)              -0.0
num__Curricular units 1st sem (evaluations)           -0.0
num__Curricular units 1st sem (approved)               0.0
num__Curricular units 1st sem (grade)                 -0.0
num__Curricular units 1st sem (without evaluations)    0.0
num__Unemployment rate                                 0.0
num__Inflation rate                                    0.0
num__GDP                                              -0.0
dtype: float64


In [17]:
# feature count
print("Original feature count:", X.shape[1])
print("Processed feature count:", X_train_processed.shape[1])
# processed feature count is more -> thats normal

Original feature count: 30
Processed feature count: 230


In [18]:
# save processed data
import os

os.makedirs("artifacts", exist_ok=True)

# Save preprocessing pipeline
joblib.dump(
    preprocessor,
    "artifacts/preprocessor.joblib"
)

# Save processed features
X_train_processed.to_csv(
    "artifacts/X_train_processed.csv",
    index=False
)

X_test_processed.to_csv(
    "artifacts/X_test_processed.csv",
    index=False
)

# Save target variables
y_train.to_csv(
    "artifacts/y_train.csv",
    index=False
)

y_test.to_csv(
    "artifacts/y_test.csv",
    index=False
)

print("All preprocessing artifacts saved successfully!")

All preprocessing artifacts saved successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
# check everything
print("========== PREPROCESSING SUMMARY ==========")

print("Scenario:", PREDICTION_SCENARIO)

print("Original dataset:", X.shape)

print("Training samples:", len(X_train_processed))
print("Testing samples:", len(X_test_processed))

print("Processed features:", X_train_processed.shape[1])

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

print("\nMissing values:",
      X_train_processed.isnull().sum().sum())

print("\nPreprocessing completed successfully!")

========== PREPROCESSING SUMMARY ==========
Scenario: after_first_semester
Original dataset: (4424, 30)
Training samples: 3539
Testing samples: 885
Processed features: 230

Training target distribution:
Target
Graduate    1767
Dropout     1137
Enrolled     635
Name: count, dtype: int64

Testing target distribution:
Target
Graduate    442
Dropout     284
Enrolled    159
Name: count, dtype: int64

Missing values: 0

Preprocessing completed successfully!


In [ ]:
#Transform dataset

In [20]:
import os
import pandas as pd

artifact_files = os.listdir("artifacts")

print("Saved artifacts:")
for file in artifact_files:
    print("-", file)

X_train = pd.read_csv("artifacts/X_train_processed.csv")
X_test = pd.read_csv("artifacts/X_test_processed.csv")
y_train = pd.read_csv("artifacts/y_train.csv")
y_test = pd.read_csv("artifacts/y_test.csv")

print("\nDataset shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.iloc[:, 0].value_counts())

Saved artifacts:
- y_test.csv
- y_train.csv
- preprocessor.joblib
- X_train_processed.csv
- X_test_processed.csv

Dataset shapes:
X_train: (3539, 230)
X_test: (885, 230)
y_train: (3539, 1)
y_test: (885, 1)

Training target distribution:
Target
Graduate    1767
Dropout     1137
Enrolled     635
Name: count, dtype: int64
